# Hybrid framework DQN - Testing

In [ ]:
METHOD = "HYBRID"

In [ ]:
import os
import io
import random
import math
import torch
import json
import pickle
import contextlib
import numpy as np
import matplotlib.pyplot as plt
from operator import itemgetter
from ultralytics import YOLO 
import torch.nn.functional as F
from tqdm import tqdm
from mylib import myutils
from mylib import simsettings
from mylib import simtools
from mylib import probtools
from mylib import dqn
from mylib import yolo_patch_softmax as _
from constants import OBS_SCALE, CELL_SIDE, MAP_RESOLUTION
from constants import AGENT_HEIGHT, AGENT_RADIUS 
from constants import MAX_ITER_COEF, CONFIDENCE_THRESHOLD, LOCATION_ERROR_THRESHOLD, PSEUDO_COUNT_THRESHOLD
from constants import NUM_CLASSES, DIRICHLET_PRIOR
from constants import ACTIONS
from dotenv import load_dotenv
load_dotenv()

import habitat_sim
import habitat_sim.nav as nav
from habitat.utils.visualizations import maps
from habitat_sim.utils import common as utils

# Reload imported modules
%load_ext autoreload
%autoreload 2

# Load YOLO model
yolo_model = YOLO("yolo11x.pt")

# Initialize cuda:0 device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
with contextlib.redirect_stdout(io.StringIO()):
    yolo_model = yolo_model.to(device)

In [ ]:
# Load the JSON file for simulation
with open('simulation-data/simulations.json', 'r') as f:
    simulations = json.load(f)

# Simulation configuration 
simulation = simulations[55] # 53 or 54 would be the same; 55 test
 
# Access its fields
SCENE = simulation["scene"]
TARGET_OBJECT = simulation["target_object"]
TARGET_OBJECT_ID = simulation["target_object_id"]
REAL_TARGET_LOCATION = simulation["target_object_location"]
INDEX = simulation["index"]

# Load the JSON file for RGB camera intrinsics
with open('simulation-data/camera-intrinsics.json', 'r') as f:
    intrinsics = json.load(f)

# Load bins per class
with open('simulation-data/object-classes-bins.json', 'r') as f:
    classes_bins = json.load(f)

# Load indoor objects
with open('simulation-data/indoor-objects.json', 'r') as f:
    data = json.load(f)
    indoor_objects = [list(item.values())[0] for item in data["indoor_classes"]]

TARGET_OBJECT_CLASS_ID = indoor_objects.index(TARGET_OBJECT)

# Load Dirichlet priors
with open('simulation-data/dirichlet-alpha-priors-augmented.pkl', 'rb') as f:
    dirichlet_priors = pickle.load(f)

# Simulator configuration
dataset_config_file = os.path.join(os.getenv("AI2THOR_DATA"), "ai2thor-hab.scene_dataset_config.json")
sim_settings = {
    "seed": 1,
    "dataset": dataset_config_file,  # Scene dataset
    "scene": SCENE,  # Scene path
    "width": 1024,  # Spatial resolution of the observations
    "height": int(1024*OBS_SCALE),
    "default_agent": 0,
    "sensor_height": AGENT_HEIGHT,  # Height of sensors in meters
    "color_sensor": True,  # RGB sensor
    "depth_sensor": True,  # Depth sensor
    "enable_physics": False,  # kinematics only
}
# Initialize the simulator
cfg = simsettings.make_cfg(sim_settings)
sim = habitat_sim.Simulator(cfg)

In [ ]:
# Get the root node of the active scene graph
scene_root = sim.get_active_scene_graph().get_root_node()
scene_bb = scene_root.cumulative_bb
scene_dims = scene_bb.size()

# Define navmesh settings
navmesh_settings = simsettings.create_navmesh_settings(AGENT_HEIGHT, AGENT_RADIUS, max_climb=0.2, max_slope=45.0, include_static_objects=True)

# Recompute the navmesh for the current scene
sim.recompute_navmesh(sim.pathfinder, navmesh_settings)

# Generate the top-down map --> 1 cm per pixel
topdown_map = maps.get_topdown_map(sim.pathfinder, height=0, meters_per_pixel=MAP_RESOLUTION, draw_border=True)
topdown_map, topdown_resolution = myutils.process_raw_topdown_map(topdown_map)

# Generate the coarse map --> 30 cm per pixel (robot has radius 15 cm)
grid_map = maps.get_topdown_map(sim.pathfinder, height=0, meters_per_pixel=CELL_SIDE, draw_border=False)
grid_map, grid_resolution = myutils.process_raw_grid_map(grid_map, sim.pathfinder)

# Find free and occupied cells in the grid map
grid_free_cells, map_free_cells, world_free_coords = myutils.find_free_cells(grid_map, grid_resolution, topdown_map, topdown_resolution, sim.pathfinder)
grid_occ_cells, map_occ_cells, world_occ_coords = myutils.find_occupied_cells(grid_map, grid_resolution, topdown_map, topdown_resolution, sim.pathfinder)

# Merge free and occupied positions into a single list
grid_cells = grid_free_cells + grid_occ_cells
map_cells = map_free_cells + map_occ_cells
world_coords = world_free_coords + world_occ_coords

# Count the number of occupiable positions
num_free_cells = len(grid_free_cells)

# Build binary int grid map (0: free, 1: occupied)
grid_binary_map = np.zeros(grid_map.shape[:2], dtype=np.int32)
for x, y in grid_occ_cells:
	if 0 <= x < grid_binary_map.shape[0] and 0 <= y < grid_binary_map.shape[1]:
		grid_binary_map[x, y] = 1

In [ ]:
# Simulation parameters
MAX_ITER = int(num_free_cells * MAX_ITER_COEF)  # Maximum number of actions to perform

# Define initial clusters
num_clusters = simtools.num_cluster_centers(num_free_cells, max_clusters=num_free_cells)
cluster_map = simtools.cluster_mapping(grid_free_cells, num_clusters)
cluster_centers = simtools.get_cluster_centers(cluster_map, num_clusters)
cluster_centers_copy = cluster_centers.copy()

# Initialize belief map: NUM_CLASSES + 1 for the background 
belief_map = [[np.ones(NUM_CLASSES + 1) * DIRICHLET_PRIOR for _ in range(grid_resolution[1])] for _ in range(grid_resolution[0])]

# Initialize an agent
agent = sim.initialize_agent(sim_settings["default_agent"])

# Sample a random position (within the possible ones)
grid_position = random.choice(grid_free_cells)
idx = grid_free_cells.index(grid_position)
world_position = world_free_coords[idx]
map_position = map_free_cells[idx]

# Sample a random yaw rotation
agent_yaw = random.choice([0, 90, 180, 270])
agent_quart = myutils.yaw_to_quaternion(agent_yaw)

# Set agent state
agent_state = habitat_sim.AgentState()
agent_state.position = world_position
agent_state.rotation = agent_quart
agent.set_state(agent_state)

# Compute agent radius in both maps
min_bounds, max_bounds = sim.pathfinder.get_bounds()
x_dim = max_bounds[0] - min_bounds[0]
topdown_radius = (AGENT_RADIUS / x_dim * topdown_resolution[0])
grid_radius = (AGENT_RADIUS / x_dim * grid_resolution[0])

# Get initial agent position tuple and radius tuple
agent_radius = (topdown_radius, grid_radius)
agent_positions = (map_position, grid_position)

# Get initial observations and maps
observations = sim.get_sensor_observations(0)
rgb, depth = observations["color_sensor"], observations["depth_sensor"]

# Display the initial simulation state (maps + observations)
simtools.display_sim_state(rgb, depth, topdown_map, grid_map, agent_positions, agent_radius, agent_yaw)
simtools.display_topdown_maps_with_clusters(topdown_map, grid_map, agent_positions, agent_radius, agent_yaw, cluster_map, cluster_centers)

In [ ]:
starting_grid_positions = [[3, 28], [4, 25], [6, 27], [4, 22], [6, 7], [12, 11], [11, 24], [4, 14], [12, 4], [11, 3], [11, 29], [3, 26], [12, 6], [11, 22], [10, 27], [6, 25], [12, 4], [8, 17], [11, 22], [7, 26], [11, 28], [11, 3], [4, 12], [4, 14], [7, 28], [6, 16], [13, 13], [3, 25], [4, 26], [8, 14], [4, 8], [3, 8], [7, 14], [6, 5], [10, 15], [11, 15], [4, 9], [13, 23], [11, 23], [11, 24], [3, 17], [6, 19], [9, 15], [3, 7], [3, 27], [6, 13], [11, 20], [6, 14], [5, 20], [13, 10], [6, 5], [12, 13], [3, 6], [12, 11], [5, 6], [3, 20], [6, 7], [9, 18], [11, 7], [4, 24], [10, 24], [5, 25], [5, 17], [11, 26], [8, 19], [3, 24], [11, 19], [7, 17], [10, 27], [3, 8], [10, 26], [9, 14], [12, 6], [3, 26], [9, 16], [10, 24], [12, 13], [3, 18], [6, 24], [3, 15], [5, 7], [12, 12], [12, 3], [11, 16], [3, 19], [11, 13], [10, 18], [4, 27], [13, 5], [11, 16], [6, 26], [5, 8], [10, 19], [11, 23], [9, 17], [12, 11], [6, 14], [6, 17], [7, 26], [3, 25], ]
starting_orientations = [180, 180, 270, 180, 270, 90, 90, 90, 90, 270, 0, 180, 90, 90, 0, 90, 90, 180, 90, 270, 90, 90, 90, 0, 90, 180, 270, 90, 90, 180, 90, 0, 90, 0, 270, 90, 270, 90, 270, 270, 270, 180, 270, 0, 90, 180, 180, 0, 180, 180, 180, 270, 0, 180, 90, 270, 180, 0, 90, 180, 90, 270, 180, 270, 0, 90, 270, 0, 270, 0, 90, 180, 270, 0, 270, 270, 90, 0, 0, 90, 0, 180, 180, 270, 0, 270, 270, 180, 180, 0, 90, 270, 0, 90, 90, 180, 180, 90, 0, 180, ]

In [ ]:
# Initialize DQN + target network
policy_net = dqn.ObjectSearchQNetwork(in_channels=4, num_actions=num_free_cells).to(device)

if os.path.exists("dqn_policy.pth"):
    policy_net.load_state_dict(torch.load("dqn_policy.pth"))
    print("Loaded pretrained model from dqn_policy.pth")

# Track performance
episode_lengths = []
episode_location_errors = []
episode_travelled_distances = []
episode_success_flags = []  
episode_rewards = []

In [ ]:
# Main training loop
for index, (GRID_POSITION, AGENT_YAW) in enumerate(zip(starting_grid_positions, starting_orientations)):

    target_grid_position = None
    
    # Metrics init
    target_found = False
    num_actions = 0
    travelled_distance = 0.0
    location_error = float("inf")
    episode_reward = 0.0

    # Get epsilon for exploration
    epsilon = 0.1

    # Reset belief map
    belief_map = [[np.ones(NUM_CLASSES + 1) * DIRICHLET_PRIOR for _ in range(grid_resolution[1])] for _ in range(grid_resolution[0])]

    # Reset cluster centers 
    num_clusters = simtools.num_cluster_centers(num_free_cells, max_clusters=num_free_cells)

    # Init agent state
    agent = sim.initialize_agent(sim_settings["default_agent"])

    # Sample a random position (within the possible ones)
    grid_position = GRID_POSITION
    idx = grid_free_cells.index(grid_position)
    world_position = world_free_coords[idx]
    map_position = map_free_cells[idx]

    # Sample a random yaw rotation
    agent_yaw = AGENT_YAW
    agent_quart = myutils.yaw_to_quaternion(agent_yaw)

    # Set agent state
    agent_state = habitat_sim.AgentState()
    agent_state.position = world_position
    agent_state.rotation = agent_quart
    agent.set_state(agent_state)

    # Set observations
    state = dqn.build_fullmap_obs(
        np.array(np.array(belief_map, dtype=np.float32)), 
        TARGET_OBJECT_CLASS_ID, 
        grid_binary_map, 
        tuple(grid_position), 
        NUM_CLASSES+1
    )

    # Plot
    # simtools.display_topdown_maps_with_clusters(topdown_map, grid_map, agent_positions, agent_radius, agent_yaw, cluster_map, cluster_centers)

    # Run episode
    while not target_found and num_actions < MAX_ITER and num_clusters <= num_free_cells:

        # Compute clusters
        cluster_map = simtools.cluster_mapping(grid_free_cells, num_clusters)
        cluster_centers = simtools.get_cluster_centers(cluster_map, num_clusters)
        cluster_centers_copy = cluster_centers.copy()

        # Compute valid actions and mask
        mask = np.zeros(num_free_cells, dtype=np.float32)
        for cluster_center in cluster_centers:
            idx = grid_free_cells.index(cluster_center)
            mask[idx] = 1.0

        while not target_found and num_actions < MAX_ITER and mask.sum() > 0:

            # Reset reward
            reward = 0.0

            # Select action using epsilon-greedy 
            q_action = dqn.select_action(policy_net, state, mask, epsilon)
            cluster_center = grid_free_cells[q_action]

            # Select next valid mask (after taking the action)
            next_mask = mask.copy()
            next_mask[q_action] = 0.0  # Remove the selected cluster center from the next valid mask

            # # Print action to debug
            # print(f"\nCurrent position: {grid_position}")
            # print(f"Cluster center selected: {cluster_center}.")

            # Compute path to cluster center
            _, path = simtools.compute_path(grid_position, cluster_center, grid_free_cells)

            # Move through the path
            while (path) and (num_actions < MAX_ITER) and (not target_found):
                
                # Compute actions to move to next path cell
                action_list = simtools.compute_relative_actions(grid_position, agent_yaw, path[0], ACTIONS)

                # Perform the rotation action, if needed
                for action in action_list:

                    # Perform action
                    grid_position, agent_yaw = simtools.perform_action(action, grid_position, agent_yaw)
                    idx = grid_free_cells.index(grid_position)
                    map_position, world_position = map_free_cells[idx], world_free_coords[idx]
                    agent_positions = (map_position, grid_position)
                    agent_quart = myutils.yaw_to_quaternion(agent_yaw)

                    # Update metrics
                    num_actions += 1
                    travelled_distance += simtools.compute_travelled_distance(agent_state.position, world_position)

                    # Update reward
                    reward -= 0.01  # Small step penalty to encourage efficiency

                    # Update agent state
                    agent_state.position = world_position
                    agent_state.rotation = agent_quart
                    agent.set_state(agent_state)

                    # Get observations
                    obs = sim.get_sensor_observations(0)
                    rgb, depth = obs["color_sensor"], obs["depth_sensor"]

                    # YOLO Prediction
                    results = yolo_model.predict(source=rgb[:,:,:3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
                    detections = simtools.parse_yolo_detections(results)
                    simtools.merge_rgb_yolo_outputs(rgb, detections)
                    target_found, target_bbox = simtools.was_target_found(TARGET_OBJECT_ID, detections, CONFIDENCE_THRESHOLD)
                    
                    # Initialize processed cells
                    processed_cells = []

                    # Process the observations
                    for det in detections:
                        box, class_id, confidence, name, prob_vector = itemgetter('box', 'class_id', 'confidence', 'name', 'prob_vector')(det)
                        scale = myutils.compute_bbox_scale(box, rgb)

                        # Center of bbox and depth value
                        center_x, center_y = simtools.get_box_center(box)
                        depth_value = depth[center_y, center_x]

                        # Project to real world, grid and map coords
                        camera_world_position = simtools.get_camera_pos_from_agent_pos(world_position, AGENT_HEIGHT)
                        object_position = simtools.compute_real_world_position_from_pixel(camera_world_position, agent_quart, depth_value, center_x, center_y, intrinsics)
                        object_map_position, object_grid_position = simtools.get_2d_coords(object_position, topdown_resolution, grid_resolution, sim.pathfinder)

                        # Check if projected cell is outside free grid cells
                        if object_grid_position in grid_free_cells:
                            object_grid_position = simtools.get_closest_grey_cell(tuple(object_grid_position), grid_map)

                        # Add this cell to processed cells
                        if tuple(object_grid_position) not in processed_cells:
                            processed_cells.append(tuple(object_grid_position))

                        # Likelihood vector: likelihood of each class (Rule 1)
                        likelihood_vector = probtools.compute_likelihood_vector(prob_vector, scale, dirichlet_priors, classes_bins)

                        # Kaplan Update to the belief map in that cell
                        grid_x, grid_y = object_grid_position
                        belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                    # Compute every theoretically visible occupied cell
                    rays = simtools.simulate_visibility_rays(grid_map, grid_position, agent_yaw)
                    visible_occ_cells = simtools.compute_visible_occ_cells(rays, grid_map, depth, grid_cells, world_coords, grid_position, agent_state.rotation, intrinsics)
                
                    # Go through every visible occupied cell and update the belief map
                    for cell in visible_occ_cells:
                        if tuple(cell) not in processed_cells:

                            # Compute distance to the cell
                            grid_x, grid_y = cell
                            distance = np.linalg.norm((np.array(grid_position) - np.array([grid_x, grid_y]))* CELL_SIDE)

                            # Compute the likelihood vector for the cell (Rule 2)
                            likelihood_vector = probtools.compute_background_likelihood_vector(distance, NUM_CLASSES)

                            # Kaplan update to the belief map in that cell
                            belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)
                    
                    # Check if the target object was found in the belief map
                    if not target_found:
                        target_found, target_grid_position = simtools.check_target_probability_in_entropy_map(belief_map, grid_cells, TARGET_OBJECT_CLASS_ID, PSEUDO_COUNT_THRESHOLD)

                    # Compute the target location if it was found
                    if target_found:
                        if target_grid_position is None: # Not found in the belief map
                            # Real world position
                            center_x, center_y = simtools.get_box_center(target_bbox)
                            depth_value = depth[center_y, center_x]
                            target_location = simtools.compute_real_world_position_from_pixel(agent_state.position, agent_state.rotation, depth_value, center_x, center_y, intrinsics)
                        else:
                            # Real world position from grid position
                            target_location = world_coords[grid_cells.index(target_grid_position)]

                    # Evaluating correctness of the target detection
                    if target_found:
                        # Location error
                        location_error = simtools.compute_location_error(target_location, REAL_TARGET_LOCATION)

                        # Compare with threshold
                        if location_error > LOCATION_ERROR_THRESHOLD:
                            target_found = False
                        else:
                            reward += 1.0  # Positive reward for finding the target
                            break

                # Check if target was found
                if target_found: break

                # Remove the path cell
                path.pop(0)

            # When in cluster center, rotate 3 times to get the full 360 degrees view
            entropy_map = simtools.compute_entropy_map(belief_map, grid_map)
            # simtools.display_sim_observations(rgb, depth)
            # simtools.display_topdown_and_entropy_maps(topdown_map, grid_map, entropy_map, cluster_map, cluster_centers_copy, agent_positions, agent_radius, agent_yaw)
            for i in range(3):
                # Check if target was found
                if target_found: break

                grid_position, agent_yaw = simtools.perform_action('turn_right', grid_position, agent_yaw)
                agent_quart = myutils.yaw_to_quaternion(agent_yaw)
                agent_state.rotation = agent_quart
                agent.set_state(agent_state)

                # Update metrics
                num_actions += 1

                # Update reward
                reward -= 0.01  # Small step penalty to encourage efficiency

                # Get observations
                observations = sim.get_sensor_observations(0)
                rgb = observations["color_sensor"]
                depth = observations["depth_sensor"]

                # YOLO Prediction
                results = yolo_model.predict(source=rgb[:,:,:3], device='cuda:0', conf=0.30, iou=0.40, verbose=False, max_det=10)
                detections = simtools.parse_yolo_detections(results)
                simtools.merge_rgb_yolo_outputs(rgb, detections)
                target_found, target_bbox = simtools.was_target_found(TARGET_OBJECT_ID, detections, CONFIDENCE_THRESHOLD)

                # Initialize processed cells
                processed_cells = []

                # Process the observations
                for det in detections:
                    box, class_id, confidence, name, prob_vector = itemgetter('box', 'class_id', 'confidence', 'name', 'prob_vector')(det)
                    scale = myutils.compute_bbox_scale(box, rgb)

                    # Center of bbox and depth value
                    center_x, center_y = simtools.get_box_center(box)
                    depth_value = depth[center_y, center_x]

                    # Project to real world, grid and map coords
                    camera_world_position = simtools.get_camera_pos_from_agent_pos(world_position, AGENT_HEIGHT)
                    object_position = simtools.compute_real_world_position_from_pixel(camera_world_position, agent_quart, depth_value, center_x, center_y, intrinsics)
                    object_map_position, object_grid_position = simtools.get_2d_coords(object_position, topdown_resolution, grid_resolution, sim.pathfinder)

                    # Check if projected cell is outside free grid cells
                    if object_grid_position in grid_free_cells:
                        object_grid_position = simtools.get_closest_grey_cell(tuple(object_grid_position), grid_map)

                    # Add this cell to processed cells
                    if tuple(object_grid_position) not in processed_cells:
                        processed_cells.append(tuple(object_grid_position))

                    # Likelihood vector: likelihood of each class (Rule 1)
                    likelihood_vector = probtools.compute_likelihood_vector(prob_vector, scale, dirichlet_priors, classes_bins)

                    # Kaplan Update to the belief map in that cell
                    grid_x, grid_y = object_grid_position
                    belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                # Compute every theoretically visible occupied cell
                rays = simtools.simulate_visibility_rays(grid_map, grid_position, agent_yaw)
                visible_occ_cells = simtools.compute_visible_occ_cells(rays, grid_map, depth, grid_cells, world_coords, grid_position, agent_state.rotation, intrinsics)

                # Go through every visible occupied cell and update the belief map
                for cell in visible_occ_cells:
                    if tuple(cell) not in processed_cells:

                        # Compute distance to the cell
                        grid_x, grid_y = cell
                        distance = np.linalg.norm((np.array(grid_position) - np.array([grid_x, grid_y]))* CELL_SIDE)

                        # Compute the likelihood vector for the cell (Rule 2)
                        likelihood_vector = probtools.compute_background_likelihood_vector(distance, NUM_CLASSES)

                        # Kaplan update to the belief map in that cell
                        belief_map[grid_x][grid_y] = probtools.kaplan_update(belief_map[grid_x][grid_y], likelihood_vector)

                if not target_found:
                    target_found, target_grid_position = simtools.check_target_probability_in_entropy_map(belief_map, grid_cells, TARGET_OBJECT_CLASS_ID, PSEUDO_COUNT_THRESHOLD)

                # Compute the target location if it was found
                if target_found:
                    if target_grid_position is None: # Not found in the belief map
                        # Real world position
                        center_x, center_y = simtools.get_box_center(target_bbox)
                        depth_value = depth[center_y, center_x]
                        target_location = simtools.compute_real_world_position_from_pixel(agent_state.position, agent_state.rotation, depth_value, center_x, center_y, intrinsics)
                    else:
                        # Real world position from grid position
                        target_location = world_coords[grid_cells.index(target_grid_position)]

                # Evaluating correctness of the target detection
                if target_found:
                    # Location error
                    location_error = simtools.compute_location_error(target_location, REAL_TARGET_LOCATION)

                    # Compare with threshold
                    if location_error > LOCATION_ERROR_THRESHOLD:
                        target_found = False
                    else:
                        reward += 1.0  # Positive reward for finding the target
                        break

            # Compute next observation
            next_state = dqn.build_fullmap_obs(
                np.array(np.array(belief_map, dtype=np.float32)), 
                TARGET_OBJECT_CLASS_ID, 
                grid_binary_map, 
                tuple(grid_position), 
                NUM_CLASSES+1
            )

            # Is it done?
            done = target_found

            # print(f"Action: {action}, Reward: {reward:.2f}, Done: {done}")
            # print(f"Clusters before: {mask.sum()}, after: {next_mask.sum()}")

            state = next_state
            mask = next_mask
            episode_reward += reward

        # Update clusters
        num_clusters *= 2  # Double the number of clusters


    # Display simulation result and metrics
    if target_found:
        print(f"\nTarget object <{TARGET_OBJECT}> found after {num_actions} actions!")
        print(f"Found location: {target_location}")
    else:
        print(f"\nTarget object <{TARGET_OBJECT}> not found after {num_actions} actions!")

    print(f"Ground-truth location: {REAL_TARGET_LOCATION}")
    print(f"Number of actions: {num_actions}")
    print(f"Travelled distance: {travelled_distance:.2f} m")
    print(f"Computed location error: {location_error:.3f} m")

    print(f"Episode reward: {episode_reward:.2f}")
    simtools.display_sim_state(rgb, depth, topdown_map, grid_map, agent_positions, agent_radius, agent_yaw)

    # Append metrics
    episode_lengths.append(num_actions)
    episode_location_errors.append(location_error)
    episode_travelled_distances.append(travelled_distance)
    episode_success_flags.append(target_found)
    episode_rewards.append(episode_reward)

In [ ]:
import numpy as np

# Pre-process
num_actions_epochs = np.array(episode_lengths)
travelled_distance_epochs = np.array(episode_travelled_distances)
success_epochs = np.array(episode_success_flags)
location_error_epochs = np.array(episode_location_errors)

# Counting
num_total_epochs = len(num_actions_epochs)
num_success_epochs = np.sum(success_epochs)

# Metrics on all runs
success_rate = num_success_epochs / num_total_epochs * 100

# Metrics on successful runs (Averages)
success_avg_travelled_distance = np.sum(travelled_distance_epochs[success_epochs]) / num_success_epochs
success_avg_num_actions = np.sum(num_actions_epochs[success_epochs]) / num_success_epochs
success_avg_location_error = np.sum(location_error_epochs[success_epochs]) / num_success_epochs

# Metrics on successful runs (Stds)
success_std_travelled_distance = np.std(travelled_distance_epochs[success_epochs])
success_std_num_actions = np.std(num_actions_epochs[success_epochs])
success_std_location_error = np.std(location_error_epochs[success_epochs])

#Print final metrics
print("\n\nMETRICS:\n")
print(f"Total number of epochs: {num_total_epochs}")
print(f"Number of successful epochs: {num_success_epochs}")
print(f"\nSuccess rate: {success_rate:.2f}%")
print(f"Average number of actions (successful epochs): {success_avg_num_actions:.2f}")
print(f"Average travelled distance (successful epochs): {success_avg_travelled_distance:.2f} m")
print(f"Average location error (successful epochs): {success_avg_location_error:.3f} m")
print(f"\nStd travelled distance (successful epochs): {success_std_travelled_distance:.2f} m")
print(f"Std number of actions (successful epochs): {success_std_num_actions:.2f}")
print(f"Std location error (successful epochs): {success_std_location_error:.3f} m")

# Number of actions max, min and std
print(f"\nMax number of actions (epochs): {np.max(num_actions_epochs)}")
print(f"Min number of actions (epochs): {np.min(num_actions_epochs)}")
print(f"Std number of actions (epochs): {np.std(num_actions_epochs)}")    

In [ ]:
metrics_file = 'results/metrics.json'

# Load existing metrics if the file exists, otherwise start with an empty list
if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        existing_metrics = json.load(f)
else:
    existing_metrics = []

# New metrics entry (convert numpy types to native Python types)
def to_python_type(val):
    if hasattr(val, "item"):
        return val.item()
    return val

new_metrics = {
    "simulation_index": to_python_type(INDEX),
    "scene": SCENE,
    "target_object": TARGET_OBJECT,
    "search_method": METHOD,
    "confidence_threshold": to_python_type(CONFIDENCE_THRESHOLD),
    "max_iter_coefficient": to_python_type(MAX_ITER_COEF),
    "location_error_threshold": to_python_type(LOCATION_ERROR_THRESHOLD),
    "pseudo_count_threshold": to_python_type(PSEUDO_COUNT_THRESHOLD),
    "num_total_epochs": to_python_type(num_total_epochs),
    "num_success_epochs": to_python_type(num_success_epochs),
    "success_rate": to_python_type(success_rate),
    "success_avg_num_actions": to_python_type(success_avg_num_actions),
    "success_avg_travelled_distance": to_python_type(success_avg_travelled_distance),
    "success_avg_location_error": to_python_type(success_avg_location_error),
    "success_std_num_actions": to_python_type(success_std_num_actions),
    "success_std_travelled_distance": to_python_type(success_std_travelled_distance),
    "success_std_location_error": to_python_type(success_std_location_error),
}

# Append new metrics to the existing list
existing_metrics.append(new_metrics)

# Save updated metrics list
with open(metrics_file, 'w') as f:
    json.dump(existing_metrics, f, indent=4)

print(f"\nMetrics saved to {metrics_file}")

In [ ]:
# Save pre-processed metrics for further analysis (JSON)
preprocessed_metrics = {
    "simulation_index": to_python_type(INDEX),
    "scene": SCENE,
    "target_object": TARGET_OBJECT,
    "search_method": METHOD,
    "confidence_threshold": to_python_type(CONFIDENCE_THRESHOLD),
    "max_iter_coefficient": to_python_type(MAX_ITER_COEF),
    "location_error_threshold": to_python_type(LOCATION_ERROR_THRESHOLD),
    "pseudo_count_threshold": to_python_type(PSEUDO_COUNT_THRESHOLD),
    "num_actions_epochs": num_actions_epochs.tolist(),
    "travelled_distance_epochs": travelled_distance_epochs.tolist(),
    "success_epochs": success_epochs.tolist(),
    "location_error_epochs": location_error_epochs.tolist(),
}

os.makedirs('results', exist_ok=True)
metrics_path = 'results/preprocessed_metrics.json'

# Load existing list or start a new one, then append and save
if os.path.exists(metrics_path):
    try:
        with open(metrics_path, 'r') as f:
            existing = json.load(f)
        if not isinstance(existing, list):
            existing = [existing]
    except Exception:
        existing = []
else:
    existing = []

existing.append(preprocessed_metrics)

with open(metrics_path, 'w') as f:
    json.dump(existing, f, indent=4)

In [ ]:
sim.close()  # Close the simulator